# Baby names with Polars

This lesson uses the local `names_year.csv` dataset to practice:

- grouping rows by one or more columns;
- using `agg()` to calculate several summaries;
- sorting aggregate tables;
- selecting a metric that answers a question; and
- creating charts from summarized data.

The dataset has one row for each name-year combination. `Total` is the recorded total for that name in that year.

In [1]:
from pathlib import Path

import polars as pl
from lets_plot import *

# Configure Lets-Plot to display charts inside the notebook.
LetsPlot.setup_html()

# Display longer strings in Polars tables without cutting them off.
pl.Config.set_fmt_str_lengths(40)

polars.config.Config

## Load the local dataset

The CSV is stored in the course-level `Datasets` folder. It contains state columns in addition to `name`, `year`, and `Total`. This lesson loads only the three columns needed for the questions, which reduces memory use and avoids unnecessary processing.

`schema_overrides` tells Polars which data types to use. `Total` is read as `Float64` because the source file contains some fractional values.

In [2]:
# Move up from Week05/Lesson1 and then into the shared Datasets folder.
DATA_PATH = Path("../../Datasets/names_year.csv")

# Read only the columns needed in this lesson.
names = pl.read_csv(
    DATA_PATH,
    columns=["name", "year", "Total"],
    schema_overrides={
        "year": pl.Int64,
        "Total": pl.Float64,
    },
)

# Display the first five rows.
names.head()

name,year,Total
str,i64,f64
"""Aaden""",2005,5.0
"""Aaden""",2007,98.0
"""Aaden""",2008,939.0
"""Aaden""",2009,1242.0
"""Aaden""",2010,414.0


In [3]:
# shape returns (number of rows, number of columns).
print("Shape:", names.shape)

# schema displays every loaded column and its Polars data type.
print("Schema:", names.schema)

# Calculate the earliest year, latest year, and number of distinct names.
names.select(
    pl.col("year").min().alias("first_year"),
    pl.col("year").max().alias("last_year"),
    pl.col("name").n_unique().alias("distinct_names"),
)

Shape: (393384, 3)
Schema: Schema({'name': String, 'year': Int64, 'Total': Float64})


first_year,last_year,distinct_names
i64,i64,u32
1910,2015,7354


## Start with a question

Before grouping, state what one row in the answer should represent.

For the question “How many rows are recorded in each year?”, one answer row should represent one year. Therefore, group by `year` and count the rows in each group.

In [4]:
rows_per_year = (
    names
    # Create one group for every year.
    .group_by("year")
    # Count the number of rows in each year group.
    .len(name="number_of_rows")
    # Put the years in chronological order.
    .sort("year")
)

rows_per_year

year,number_of_rows
i64,u32
1910,1506
1911,1570
1912,1914
1913,2038
1914,2196
…,…
2011,5396
2012,5337
2013,5245


In [6]:
# Plot the number of name-year rows recorded in each year.
(
    ggplot(
        rows_per_year,
        aes(x="year", y="number_of_rows"),
    )
    # Connect the yearly observations to show the long-term pattern.
    + geom_line(color="#2F6B9A", size=1.2)
    # Mark each yearly observation.
    + geom_point(color="#2F6B9A", size=2.0)
    # Show one labeled tick every ten years.
    + scale_x_continuous(breaks=list(range(1910, 2016, 10)), format="")
    # Add comma separators to the row counts.
    + scale_y_continuous(format=",")
    + labs(
        title="Number of names recorded each year",
        subtitle="Each row represents one name-year combination",
        x="Year",
        y="Number of rows",
    )
    + ggsize(900, 500)
    + theme_minimal()
)

## Use `agg()` for several summaries

`group_by()` defines the groups. `agg()` calculates each expression separately inside those groups.

The next table answers two questions for each year:

- How many distinct names were recorded?
- What is the sum of `Total` for the year?

In [7]:
year_summary = (
    names
    # Split the rows into year groups.
    .group_by("year")
    # Calculate both expressions inside every year group.
    .agg(
        pl.col("name").n_unique().alias("number_of_names"),
        pl.col("Total").sum().alias("annual_total"),
    )
    # Arrange the result chronologically.
    .sort("year")
)

year_summary.tail(10)

year,number_of_names,annual_total
i64,u32,f64
2006,5626,3.121052e6
2007,5614,3.146999e6
2008,5575,3.071651e6
2009,5514,2.98398e6
2010,5435,2886132.5
2011,5396,2849724.5
2012,5337,2.845152e6
2013,5245,2.836556e6
2014,5208,2877090.5


## Summarize by name

Changing the grouping column changes what each answer row represents. In the next table, one row represents one name across all available years.

In [8]:
name_summary = (
    names
    # Create one group for every distinct name.
    .group_by("name")
    .agg(
        # Add the yearly totals for an all-years total.
        pl.col("Total").sum().alias("all_years_total"),
        # Count the number of years in which the name appears.
        pl.col("year").n_unique().alias("years_recorded"),
        # Calculate the average yearly total for the name.
        pl.col("Total").mean().round(1).alias("average_yearly_total"),
        # Find the largest value the name reached in a single year.
        pl.col("Total").max().alias("largest_yearly_total"),
    )
    # Put the largest all-years total first.
    .sort("all_years_total", descending=True)
)

name_summary.head(15)

name,all_years_total,years_recorded,average_yearly_total,largest_yearly_total
str,f64,u32,f64,f64
"""John""",3.269866e6,106,30847.8,59993.5
"""James""",3.237006e6,106,30537.8,59672.5
"""Robert""",3.151493e6,106,29731.1,60812.5
"""William""",2.846782e6,106,26856.4,52877.0
"""Michael""",2.797302e6,106,26389.6,62488.0
…,…,…,…,…
"""Charles""",1.743807e6,106,16451.0,32526.0
"""Daniel""",1.534743e6,106,14478.7,26554.0
"""Christopher""",1467008.5,106,13839.7,37116.0


## Filter before grouping

Filtering changes which rows are allowed to enter the groups. The following example limits the analysis to 2000 through 2015 before calculating each name’s total during that period.

In [12]:
recent_name_summary = (
    names
    # Keep only rows from 2000 through 2015.
    .filter(pl.col("year").is_between(2000, 2015))
    # Create one group for every name in the filtered rows.
    .group_by("name")
    # Add the yearly totals within each name group.
    .agg(
        pl.col("Total").sum().alias("total_2000_to_2015")
    )
    # Put the largest period total first.
    .sort("total_2000_to_2015", descending=True)
)

recent_name_summary.head(10)

name,total_2000_to_2015
str,f64
"""Jacob""",375524.0
"""Michael""",309462.0
"""Emma""",297783.0
"""Ethan""",297014.0
"""Emily""",294878.0
"""William""",293713.5
"""Matthew""",293111.5
"""Joshua""",290228.5
"""Andrew""",267284.5


## Find the most common names in one year

Because the dataset already has one row per name-year combination, finding the largest names in one year requires filtering and sorting rather than another aggregation.

In [13]:
top_names_2015 = (
    names
    # Keep only rows from 2015.
    .filter(pl.col("year") == 2015)
    # Put the largest Total values first.
    .sort("Total", descending=True)
    # Keep the first ten rows after sorting.
    .head(10)
)

top_names_2015

name,year,Total
str,i64,f64
"""Emma""",2015,20355.0
"""Olivia""",2015,19553.0
"""Liam""",2015,18281.0
"""Sophia""",2015,17327.0
"""Ava""",2015,16286.0
"""William""",2015,15809.0
"""Isabella""",2015,15504.0
"""Jacob""",2015,15178.0
"""Ethan""",2015,14991.0


## Create a bar chart from the sorted table

The plot uses the ten-row summary rather than all 393,384 source rows. `coord_flip()` places names on the vertical axis so longer labels remain readable.

In [14]:
top_names_chart_data = top_names_2015.with_columns(
    # Create comma-formatted labels for the bars.
    pl.col("Total")
    .round(0)
    .cast(pl.Int64)
    .cast(pl.String)
    .alias("total_label")
)

(
    ggplot(
        top_names_chart_data,
        aes(x="name", y="Total"),
    )
    + geom_bar(stat="identity", fill="#2F6B9A", width=0.72)
    + geom_text(aes(label="total_label"), hjust=-0.15, size=9)
    + coord_flip()
    + scale_y_continuous(limits=[0, 23000], format=",")
    + labs(
        title="Ten names with the largest totals in 2015",
        x="Name",
        y="Total",
    )
    + ggsize(850, 550)
    + theme_minimal()
)

## Compare trends for selected names

`is_in()` keeps rows whose value appears in a list. The result is sorted before plotting so each line moves through time in the correct order.

In [15]:
selected_names = ["Emma", "Olivia", "Liam", "Noah"]

selected_name_trends = (
    names
    # Keep the selected names and years beginning in 1990.
    .filter(
        pl.col("name").is_in(selected_names),
        pl.col("year") >= 1990,
    )
    # Sort each name's observations chronologically.
    .sort(["name", "year"])
)

(
    ggplot(
        selected_name_trends,
        aes(x="year", y="Total", color="name"),
    )
    + geom_line(size=1.2)
    + labs(
        title="Recorded totals for selected names",
        subtitle="1990 through 2015",
        x="Year",
        y="Total",
        color="Name",
    )
    + ggsize(850, 500)
    + theme_minimal()
)

# In-class problems

Complete each problem with one Polars method chain. Before coding, say what one row in your answer should represent.

## Problem 1: yearly sample size

Calculate the number of rows recorded for every year. Sort the result from the largest sample size to the smallest. Save the table as `problem_1`.

In [ ]:
# TODO: Use group_by(), len(), and sort().
problem_1 = None
problem_1

## Problem 2: largest names in 1990

Find the ten names with the largest `Total` values in 1990. Save the table as `problem_2`.

In [ ]:
# TODO: Filter, sort, and keep ten rows.
problem_2 = None
problem_2

## Problem 3: summary for every name

For each name, calculate its all-years total, average yearly total, largest yearly total, and number of years recorded. Sort from largest all-years total to smallest. Save the table as `problem_3`.

In [ ]:
# TODO: Use group_by(), agg(), aliases, and sort().
problem_3 = None
problem_3

## Problem 4: names with consistent records

Using your result from Problem 3, keep only names recorded in at least 100 different years. Sort the remaining names by average yearly total from highest to lowest. Save the table as `problem_4`.

In [ ]:
# TODO: Filter and sort problem_3.
problem_4 = None
problem_4

## Closing checklist

When answering an aggregation question, check that:

- the grouping columns match what one row in the answer should represent;
- every aggregation has a clear alias;
- the metric matches the question;
- the sorting makes the result easy to interpret; and
- the chart uses the summarized table rather than the full source data.

# In-class answers

The answer cells are collapsed by default. Click a cell's source area or expand control to reveal and run the solution.

In [ ]:
# Answer 1: Count rows in every year group and rank the sample sizes.
problem_1 = (
    names
    # Create one group for each year.
    .group_by("year")
    # Count the rows in every year group.
    .len(name="number_of_rows")
    # Put the largest yearly sample first.
    .sort("number_of_rows", descending=True)
)

problem_1

In [17]:
# Answer 2: Limit the data to 1990 and rank names by Total.
problem_2 = (
    names
    # Keep only observations from 1990.
    .filter(pl.col("year") == 1990)
    # Put the largest Total values first.
    .sort("Total", descending=True)
    # Keep the first ten rows after sorting.
    .head(10)
)

problem_2

name,year,Total
str,i64,f64
"""Michael""",1990,44043.0
"""Jessica""",1990,39941.5
"""Matthew""",1990,38202.0
"""Christopher""",1990,37116.0
"""Joshua""",1990,33545.0
"""Brittany""",1990,32562.5
"""Amanda""",1990,30603.0
"""Ashley""",1990,28726.5
"""Andrew""",1990,28524.5


In [ ]:
# Answer 3: Calculate four summaries inside every name group.
problem_3 = (
    names
    # Create one group for each distinct name.
    .group_by("name")
    .agg(
        # Add the name's totals from all available years.
        pl.col("Total").sum().alias("all_years_total"),
        # Calculate the name's average yearly total.
        pl.col("Total").mean().round(1).alias("average_yearly_total"),
        # Find the name's largest value in a single year.
        pl.col("Total").max().alias("largest_yearly_total"),
        # Count the number of distinct years containing the name.
        pl.col("year").n_unique().alias("years_recorded"),
    )
    # Put the largest all-years total first.
    .sort("all_years_total", descending=True)
)

problem_3

In [ ]:
# Answer 4: Keep names with records in at least 100 different years.
problem_4 = (
    problem_3
    # Keep rows whose number of recorded years meets the requirement.
    .filter(pl.col("years_recorded") >= 100)
    # Rank the remaining names by average yearly total.
    .sort("average_yearly_total", descending=True)
)

problem_4